In [5]:
from importlib import reload
import os
import random
import sys

import numpy as np
from sklearn.preprocessing import StandardScaler
sys.path.append(os.path.abspath(os.path.join('..')))
import models

reload(models)

import torch
import pandas as pd
import mlflow
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
from datetime import datetime

from models import HybridModel
from utils import mol_to_graph, MLFlowManager, train_hybrid_model, evaluate_hybrid_model
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*') # type: ignore


import logging

logging.basicConfig(
    filename="debug_model.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    filemode="w"
)

logger = logging.getLogger(__name__)

def log_regression_plots(y_true, y_pred, run_name):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
    plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], '--r', lw=2)
    plt.xlabel("Actual pIC50")
    plt.ylabel("Predicted pIC50")
    plt.title(f"Regression Fit - {run_name}")
    plot_path = "pred_vs_actual.png"
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_one_run(params, train_loader, test_loader, device, run_name):
    model = HybridModel(
        num_node_features=4,
        num_extra_features=num_features,
        hidden_channels=params["hidden_channels"]
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=params["lr"]
    )

    best_r2 = float("-inf")

    with mf.start_run(run_name):

        for epoch in range(params["epochs"]):

            loss = train_hybrid_model(
                model,
                train_loader,
                optimizer,
                device,
                epoch
            )

            r2, mae, _, _ = evaluate_hybrid_model(
                model,
                test_loader,
                device
            )

            mlflow.log_metric("train_mse", loss, step=epoch)
            mlflow.log_metric("val_r2", r2, step=epoch)
            mlflow.log_metric("val_mae", mae, step=epoch)

            if r2 > best_r2:
                best_r2 = r2

        return best_r2
    
def process_row(row):
    scaled_features = scaler.transform(row[features].values.reshape(1, -1)).flatten()
    return mol_to_graph(row['canonical_smiles'], row['pic50'], scaled_features)

seed_everything(42)

features = [
    'alogp', 'psa', 'hba', 'hbd', 'num_ro5_violations', 'qed_weighted',
    'logP_over_PSA', 'HBA_HBD_sum'
]
num_features = len(features)
parquet_path = "parquets/subset_50k_stratified.parquet"
df = pd.read_parquet(parquet_path)
dataset_path = "/home/pkuszn/repos/WSzI/src/notebooks/data/chembl_dataset.pt"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

scaler = StandardScaler()
scaler.fit(train_df[features])

print("Processing Train...")
train_data = Parallel(n_jobs=-1)(delayed(process_row)(row) for _, row in train_df.iterrows())
train_data = [d for d in train_data if d is not None]

print("Processing Test...")
test_data = Parallel(n_jobs=-1)(delayed(process_row)(row) for _, row in test_df.iterrows())
test_data = [d for d in test_data if d is not None]

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

batch = next(iter(train_loader))

param_grid = [
    {"lr": 1e-2, "hidden_channels": 32, "epochs": 30},
    {"lr": 5e-3, "hidden_channels": 64, "epochs": 30},
    {"lr": 1e-3, "hidden_channels": 64, "epochs": 40},
    {"lr": 1e-3, "hidden_channels": 128, "epochs": 40},
    {"lr": 5e-4, "hidden_channels": 128, "epochs": 50},
]

model = HybridModel(num_node_features=4, num_extra_features=num_features, hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

mf = MLFlowManager(experiment_name="ChEMBL_HybridModel_Scaffold_Split")

now = str(int(datetime.now().timestamp()))
run_name = f"Hybrid_Run_{now}"
print("Starting Training...")

best_config = None
best_score = float("-inf")
for i, params in enumerate(param_grid):

    run_name = f"Hybrid_tune_{i}_{int(datetime.now().timestamp())}"

    print(f"\nRunning config {i+1}/{len(param_grid)}: {params}")

    score = train_one_run(
        params,
        train_loader,
        test_loader,
        device,
        run_name
    )

    if score > best_score:
        best_score = score
        best_config = params

mlflow.log_metrics({"best_r2": best_score})
print("\nBEST CONFIG:", best_config)
print("BEST R2:", best_score)

model_save_path = f"model_{run_name}_weights.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Saved weight to {model_save_path}")
mlflow.log_artifact(model_save_path)

Processing Train...


/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, b

Processing Test...


/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, b

Starting Training...

Running config 1/5: {'lr': 0.01, 'hidden_channels': 32, 'epochs': 30}
Epoch 0 | MSE: 4.9603 | MAE: 1.7945 | R2: -0.2154
Epoch 1 | MSE: 3.9039 | MAE: 1.6543 | R2: 0.0434
Epoch 2 | MSE: 3.6965 | MAE: 1.6152 | R2: 0.0942
Epoch 3 | MSE: 3.6787 | MAE: 1.6199 | R2: 0.0986
Epoch 4 | MSE: 3.6522 | MAE: 1.6141 | R2: 0.1051
Epoch 5 | MSE: 3.6271 | MAE: 1.6069 | R2: 0.1112
Epoch 6 | MSE: 3.6157 | MAE: 1.6046 | R2: 0.1140
Epoch 7 | MSE: 3.5935 | MAE: 1.5983 | R2: 0.1195
Epoch 8 | MSE: 3.5872 | MAE: 1.5961 | R2: 0.1210
Epoch 9 | MSE: 3.5639 | MAE: 1.5928 | R2: 0.1267
Epoch 10 | MSE: 3.5643 | MAE: 1.5915 | R2: 0.1266
Epoch 11 | MSE: 3.5559 | MAE: 1.5876 | R2: 0.1287
Epoch 12 | MSE: 3.5328 | MAE: 1.5823 | R2: 0.1344
Epoch 13 | MSE: 3.5161 | MAE: 1.5767 | R2: 0.1385
Epoch 14 | MSE: 3.5022 | MAE: 1.5733 | R2: 0.1419
Epoch 15 | MSE: 3.4941 | MAE: 1.5707 | R2: 0.1439
Epoch 16 | MSE: 3.4742 | MAE: 1.5659 | R2: 0.1487
Epoch 17 | MSE: 3.4784 | MAE: 1.5666 | R2: 0.1477
Epoch 18 | MSE: 3

KeyboardInterrupt: 